# Download HF Model and Save it to S3

To save this model so that you can use it from various locations, including other notebooks or the model server, upload it to s3-compatible storage.

In [1]:
!pip install huggingface_hub
!pip install boto3 botocore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.1/464.1 kB 7.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import boto3
import os
import requests
import botocore
from botocore.exceptions import NoCredentialsError
from huggingface_hub import snapshot_download

# AWS Configuration
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more data connection variables are empty. Please check your data connection to an S3 bucket.")

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)

# Configuration
s3_directory = 'granite-3b-code-instruct-2k'  # Ensure no trailing slash
huggingface_model = 'ibm-granite/granite-3b-code-instruct-2k'  # Replace with actual model name
local_model_dir = 'hf_model_download'  # Local directory to store model

# Create directory in S3 (AWS S3 does not have real folders, so we use an empty object as a marker)
def create_s3_directory(bucket, directory):
    bucket.put_object(Key=f"{directory}/")
    print(f"Created directory {directory} in bucket {bucket_name}")

# Download entire model from Hugging Face
def download_model(model_name, local_dir):
    model_path = snapshot_download(repo_id=model_name, local_dir=local_dir)
    print(f"Model downloaded to {model_path}")
    return model_path

# Upload model files to S3
def upload_to_s3(bucket, directory, local_dir):
    try:
        for root, _, files in os.walk(local_dir):
            for file in files:
                local_file_path = os.path.join(root, file)
                relative_path = os.path.relpath(local_file_path, local_dir).replace("\\", "/")  # Ensure forward slashes
                s3_file_path = f"{directory}/{relative_path}".lstrip('/')  # Remove leading slash if any
                bucket.upload_file(local_file_path, s3_file_path)
                print(f"Uploaded {local_file_path} to s3://{bucket_name}/{s3_file_path}")
    except NoCredentialsError:
        print("AWS credentials not found.")

# Execute the steps
create_s3_directory(bucket, s3_directory)
model_path = download_model(huggingface_model, local_model_dir)
upload_to_s3(bucket, s3_directory, model_path)


Created directory granite-3b-code-instruct-2k in bucket openshift-ai


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model downloaded to /opt/app-root/src/hf_model_download
Uploaded /opt/app-root/src/hf_model_download/config.json to s3://openshift-ai/granite-3b-code-instruct-2k/config.json
Uploaded /opt/app-root/src/hf_model_download/generation_config.json to s3://openshift-ai/granite-3b-code-instruct-2k/generation_config.json
Uploaded /opt/app-root/src/hf_model_download/.gitattributes to s3://openshift-ai/granite-3b-code-instruct-2k/.gitattributes
Uploaded /opt/app-root/src/hf_model_download/README.md to s3://openshift-ai/granite-3b-code-instruct-2k/README.md
Uploaded /opt/app-root/src/hf_model_download/special_tokens_map.json to s3://openshift-ai/granite-3b-code-instruct-2k/special_tokens_map.json
Uploaded /opt/app-root/src/hf_model_download/model.safetensors.index.json to s3://openshift-ai/granite-3b-code-instruct-2k/model.safetensors.index.json
Uploaded /opt/app-root/src/hf_model_download/tokenizer_config.json to s3://openshift-ai/granite-3b-code-instruct-2k/tokenizer_config.json
Uploaded /opt/ap

In [6]:
import shutil

# Remove local directory and its contents
def remove_local_directory(model_dir):
    if os.path.exists(model_dir):
        shutil.rmtree(model_dir)
        print(f"Removed local directory: {model_dir}")
    else:
        print(f"Local directory not found: {model_dir}")
        
remove_local_directory(local_model_dir)


Removed local directory: hf_model_download
